In [1]:
import os
import re
import json
import math
import time
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from groq import Groq
from pathlib import Path
from dotenv import load_dotenv


import pandas as pd
from pprint import pprint

from langchain_community.document_loaders import PyPDFLoader

d:\AIT_NLP\NLP\A6_RAG_Techniques\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Task 1: Source Discovery & Data Preparation

Assigned chapter: Chapter 8  
Topic: Transformers

In [39]:
pdf_path = "D:\\AIT_NLP\\nlp\\A6_RAG_Techniques\\Chapter-8 RAG assignment.pdf"

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("Number of pages:", len(pages))
print("\nFirst 1000 characters from page 1:\n")
print(pages[0].page_content[:1000])

Number of pages: 27

First 1000 characters from page 1:

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing. Indeed, every subsequent chapter in this textbook will make use of them.
As with the previous chapter, we’ll focus for this chapter on the use of transformers
to model left-to-right (sometimes called causal or autoregressive) language model-
ing, in which we are given a sequence of input tokens and predict output tokens one
by one by conditioning on the prior context.
Layer Norm
+
Layer Norm
MultiHead
Attention
Feedforward
Softmax
Positi

Combining All the texts

In [40]:
full_text = "\n".join([page.page_content for page in pages])

print("Total characters:", len(full_text))
print(full_text[:1500])

Total characters: 76773
Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing. Indeed, every subsequent chapter in this textbook will make use of them.
As with the previous chapter, we’ll focus for this chapter on the use of transformers
to model left-to-right (sometimes called causal or autoregressive) language model-
ing, in which we are given a sequence of input tokens and predict output tokens one
by one by conditioning on the prior context.
Layer Norm
+
Layer Norm
MultiHead
Attention
Feedforward
Softmax
Positional
Unembedding
Embedding
+
resi

Cleaning text

In [41]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = text.replace("\t", " ")
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

cleaned_text = clean_text(full_text)

print("Cleaned text length:", len(cleaned_text))
print(cleaned_text[:1500])

Cleaned text length: 76763
Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing. Indeed, every subsequent chapter in this textbook will make use of them.
As with the previous chapter, we’ll focus for this chapter on the use of transformers
to model left-to-right (sometimes called causal or autoregressive) language model-
ing, in which we are given a sequence of input tokens and predict output tokens one
by one by conditioning on the prior context.
Layer Norm
+
Layer Norm
MultiHead
Attention
Feedforward
Softmax
Positional
Unembedding
Embedding
+
r

Saving Cleaned Text

In [ ]:
output_dir = Path("answer")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "chapter8_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("Saved cleaned text to", output_dir / "chapter8_cleaned.txt")

Saved cleaned text to data\chapter8_cleaned.txt


In [43]:
qa_pairs = [
    {
        "question": "What is the primary purpose of the self-attention mechanism in a transformer?",
        "ground_truth_answer": "Self-attention allows a model to build contextual representations of a token by integrating information from other tokens in the sequence.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "In the attention mechanism, what roles do the query, key, and value vectors play?",
        "ground_truth_answer": "The query represents the current token being compared, the key represents tokens used for similarity comparison, and the value contains the information that is weighted and combined in the output.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "Why is a scaling factor used in the dot product of query and key vectors?",
        "ground_truth_answer": "The dot product is scaled by the square root of the key dimension to prevent large values that could cause unstable gradients during training.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "Why do transformers use multi-head attention instead of a single attention head?",
        "ground_truth_answer": "Multi-head attention allows the model to attend to different types of relationships in the sequence simultaneously using multiple attention heads.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What components are included in a standard transformer block?",
        "ground_truth_answer": "A transformer block includes a multi-head self-attention layer, a feedforward network, residual connections, and layer normalization.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },

    {
        "question": "What is the residual stream in a transformer block?",
        "ground_truth_answer": "The residual stream is the pathway where token representations are passed through layers while each component reads from and adds its output back to the stream.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "How are positional embeddings combined with token embeddings in a transformer?",
        "ground_truth_answer": "Positional embeddings are added to token embeddings so the model can represent both the token identity and its position in the sequence.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is the architecture of the feedforward layer in a transformer block?",
        "ground_truth_answer": "The feedforward layer is a two-layer fully connected network applied independently to each token representation.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is the purpose of layer normalization in transformers?",
        "ground_truth_answer": "Layer normalization stabilizes training by normalizing activations so they have a consistent scale across the network.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "Why is masking used in the self-attention of causal language models?",
        "ground_truth_answer": "Masking prevents tokens from attending to future tokens so the model only uses previous context when predicting the next token.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What components make up the language modeling head?",
        "ground_truth_answer": "The language modeling head consists of a linear projection called the unembedding layer followed by a softmax to produce probabilities over the vocabulary.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is weight tying in transformer language models?",
        "ground_truth_answer": "Weight tying refers to sharing the same weight matrix between the token embedding layer and the final unembedding layer.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "How does top-k sampling work during text generation?",
        "ground_truth_answer": "Top-k sampling restricts the probability distribution to the k most likely tokens and randomly samples the next token from that subset.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is the intuition behind top-p or nucleus sampling?",
        "ground_truth_answer": "Top-p sampling selects the smallest set of tokens whose cumulative probability exceeds a threshold p and samples from that set.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What training objective is commonly used for large language models?",
        "ground_truth_answer": "Large language models are typically trained using cross-entropy loss to maximize the probability of the correct next token.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "How does the KV cache improve inference efficiency?",
        "ground_truth_answer": "The KV cache stores key and value vectors from previous tokens so they do not need to be recomputed during autoregressive generation.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "Why is attention sometimes called a token-mixing component?",
        "ground_truth_answer": "Attention is called token-mixing because it integrates information from other tokens into the representation of the current token.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is a decoder-only transformer model?",
        "ground_truth_answer": "A decoder-only transformer is a unidirectional model that predicts tokens autoregressively using only the decoder architecture.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is a limitation of absolute positional embeddings?",
        "ground_truth_answer": "Absolute positional embeddings may generalize poorly to positions near the maximum sequence length because those positions appear less frequently in training.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    },
    {
        "question": "What is the purpose of the logit lens tool?",
        "ground_truth_answer": "The logit lens is an interpretability method that applies the final unembedding layer to intermediate activations to analyze what the model is predicting at different layers.",
        "naive_rag_answer": "",
        "contextual_retrieval_answer": ""
    }
]

In [44]:
with open(output_dir / "qa_pairs.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print("Saved QA pairs to", output_dir / "qa_pairs_task1.json")

Saved QA pairs to data\qa_pairs_task1.json


chunking

In [45]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    
    return chunks

chunks = chunk_text(cleaned_text, chunk_size=500, overlap=50)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0][:800])

Number of chunks: 171

First chunk:

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing


embedding model

In [46]:
embedding_model_name = "BAAI/bge-small-en-v1.5"

embed_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
embed_model = AutoModel.from_pretrained(embedding_model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = embed_model.to(device)
embed_model.eval()

print("Embedding model loaded on:", device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6627.81it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded on: cpu


In [47]:
def get_embedding(text):
    inputs = embed_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = embed_model(**inputs)
    
    # CLS token embedding
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return embedding

In [48]:
VECTOR_DB = []

for chunk in tqdm(chunks, desc="Embedding chunks"):
    emb = get_embedding(chunk)
    VECTOR_DB.append((chunk, emb))

print("Vector DB size:", len(VECTOR_DB))
print("Embedding dimension:", VECTOR_DB[0][1].shape)

Embedding chunks: 100%|██████████| 171/171 [00:08<00:00, 20.26it/s]

Vector DB size: 171
Embedding dimension: (384,)


cosine similarity + retrieval

In [49]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, vector_db, top_k=3):
    query_emb = get_embedding(query)
    
    scored_chunks = []
    for chunk, emb in vector_db:
        score = cosine_similarity(query_emb, emb)
        scored_chunks.append((chunk, score))
    
    scored_chunks = sorted(scored_chunks, key=lambda x: x[1], reverse=True)
    return scored_chunks[:top_k]

test retrieval

In [50]:
test_question = "What is self-attention in a transformer?"
retrieved = retrieve(test_question, VECTOR_DB, top_k=3)

print("Question:", test_question)
print()

for i, (chunk, score) in enumerate(retrieved, 1):
    print(f"--- Retrieved chunk {i} | score={score:.4f} ---")
    print(chunk[:1000])
    print()

Question: What is self-attention in a transformer?

--- Retrieved chunk 1 | score=0.8002 ---
ding and all the outputs from all
the previous layers and blocks.
The core intuition of the transformer, and the component that distinguishes it
from the feedforward layers we saw in Chapter 6, is this multi-head attention layer,
also called a self-attention layer. Attention can be thought of as a way to build
contextual representations of a token’s meaning by attending to and integrating
information from surrounding tokens, helping the model learn how tokens relate to
each other over large span

--- Retrieved chunk 2 | score=0.7966 ---
ther linear projection WO∈ RAdv×d to reshape it, resulting in the multi-head
attention vector ai with the correct output shape [1× d] at each input i.
8.2 Transformer Blocks
The self-attention calculation lies at the core of what’s called a transformer block,
which, in addition to the self-attention layer, includes three other kinds of layers: (1)
a feedforward 

In [51]:
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


# load_dotenv(dotenv_path=Path.cwd() / ".env")

# groq_api_key = os.getenv("GROQ_API_KEY")
# if not groq_api_key:
#     raise ValueError("Please set GROQ_API_KEY in your .env file or environment.")

client = Groq(api_key=groq_api_key)
print("Groq client ready.")

Groq client ready.


answer generation function

In [52]:
def answer_question_naive(query, vector_db, top_k=3, model_name="llama-3.1-8b-instant"):
    retrieved = retrieve(query, vector_db, top_k=top_k)
    context = "\n\n".join([chunk for chunk, _ in retrieved])

    prompt = f"""
You are a helpful assistant answering questions strictly based on the provided context.

Context:
{context}

Question:
{query}

Instructions:
- Answer using only the provided context.
- If the answer is not in the context, say: "The answer is not found in the provided context."
- Keep the answer concise, around 1-3 sentences.
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    answer = response.choices[0].message.content.strip()
    return answer, retrieved

manual test of full naive RAG

In [53]:
question = "What is self-attention in a transformer?"
answer, retrieved = answer_question_naive(question, VECTOR_DB, top_k=3)

print("Question:", question)
print("\nAnswer:\n", answer)
print("\nRetrieved source chunks:\n")

for i, (chunk, score) in enumerate(retrieved, 1):
    print(f"--- Chunk {i} | score={score:.4f} ---")
    print(chunk[:1000])
    print()

Question: What is self-attention in a transformer?

Answer:
 Self-attention in a transformer can be thought of as a way to build contextual representations of a token's meaning by attending to and integrating information from surrounding tokens, helping the model learn how tokens relate to each other over large spans.

Retrieved source chunks:

--- Chunk 1 | score=0.8002 ---
ding and all the outputs from all
the previous layers and blocks.
The core intuition of the transformer, and the component that distinguishes it
from the feedforward layers we saw in Chapter 6, is this multi-head attention layer,
also called a self-attention layer. Attention can be thought of as a way to build
contextual representations of a token’s meaning by attending to and integrating
information from surrounding tokens, helping the model learn how tokens relate to
each other over large span

--- Chunk 2 | score=0.7966 ---
ther linear projection WO∈ RAdv×d to reshape it, resulting in the multi-head
attention ve

20 questions through naive RAG

In [ ]:
with open("answer/qa_pairs.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

for item in tqdm(qa_pairs, desc="Running Naive RAG"):
    question = item["question"]
    answer, _ = answer_question_naive(question, VECTOR_DB, top_k=3)
    item["naive_rag_answer"] = answer

print("Naive RAG answers generated.")

Running Naive RAG: 100%|██████████| 20/20 [00:49<00:00,  2.45s/it]

Naive RAG answers generated.


In [55]:
with open("data/qa_pairs_with_naive_rag.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print("Saved Naive RAG results.")

Saved Naive RAG results.


Contextual Retrieval

contextual enrichment function

In [56]:
def enrich_chunk(chunk, document, title="Chapter 8: Transformers", model_name="llama-3.1-8b-instant"):
    prompt = f"""
Title: {title}

Document excerpt:
{document[:4000]}

Chunk:
{chunk}

Provide brief context in 1-2 sentences explaining what this chunk discusses in relation to the full document.
Format:
This chunk from {title} discusses ...
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    context = response.choices[0].message.content.strip()
    return f"{context}\n\n{chunk}"

build contextualized chunks

In [57]:
contextual_chunks = []

for chunk in tqdm(chunks, desc="Enriching chunks for Contextual Retrieval"):
    try:
        enriched_chunk = enrich_chunk(chunk, cleaned_text, title="Chapter 8: Transformers")
        contextual_chunks.append(enriched_chunk)
    except Exception as e:
        print("Error enriching chunk:", e)
        contextual_chunks.append(chunk)  # fallback to original chunk if error happens

    time.sleep(1)  # helps with Groq free-tier rate limits

print("Number of contextual chunks:", len(contextual_chunks))
print("\nSample contextual chunk:\n")
print(contextual_chunks[0][:1200])

Enriching chunks for Contextual Retrieval:   3%|▎         | 5/171 [01:30<1:05:11, 23.56s/it]

Error enriching chunk: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km59edw5eaksgmrc9f2xwfrp` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499994, Requested 1024. Please try again in 2m55.9104s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Enriching chunks for Contextual Retrieval:   4%|▎         | 6/171 [01:31<43:48, 15.93s/it]  

Error enriching chunk: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km59edw5eaksgmrc9f2xwfrp` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499987, Requested 1024. Please try again in 2m54.7008s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Enriching chunks for Contextual Retrieval:   4%|▍         | 7/171 [01:33<30:24, 11.12s/it]

...

Error enriching chunk: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km59edw5eaksgmrc9f2xwfrp` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499631, Requested 1098. Please try again in 2m5.9712s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Enriching chunks for Contextual Retrieval:  99%|█████████▉| 170/171 [05:34<00:01,  1.10s/it]

Error enriching chunk: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km59edw5eaksgmrc9f2xwfrp` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499625, Requested 1033. Please try again in 1m53.7024s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Enriching chunks for Contextual Retrieval: 100%|██████████| 171/171 [05:35<00:00,  1.96s/it]

Number of contextual chunks: 171

Sample contextual chunk:

This chunk from Chapter 8: Transformers discusses the introduction of the transformer architecture, a standard architecture for building large language models, and its impact on the field of speech and language processing.

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
8
Transformers
“The true art of memory is the art of attention ”
Samuel Johnson, Idler #74, September 1759
In this chapter we introduce thetransformer, the standard architecture for build-
ing large language models. As we discussed in the prior chapter, transformer-based
large language models have completely changed the ﬁeld of speech and language
processing


build contextual vector database

In [58]:
contextual_chunks = []

for chunk in tqdm(chunks, desc="Enriching chunks for Contextual Retrieval"):
    try:
        enriched_chunk = enrich_chunk(chunk, cleaned_text, title="Chapter 8: Transformers")
        contextual_chunks.append(enriched_chunk)
    except Exception as e:
        print("Error enriching chunk:", e)
        contextual_chunks.append(chunk)  # fallback to original chunk if error happens

    time.sleep(1)  # helps with Groq free-tier rate limits

print("Number of contextual chunks:", len(contextual_chunks))
print("\nSample contextual chunk:\n")
print(contextual_chunks[0][:1200])

Enriching chunks for Contextual Retrieval:   0%|          | 0/171 [00:00<?, ?it/s]

Error enriching chunk: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km59edw5eaksgmrc9f2xwfrp` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499618, Requested 1039. Please try again in 1m53.5296s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Enriching chunks for Contextual Retrieval:   1%|          | 1/171 [00:01<03:06,  1.10s/it]

Error enriching chunk: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km59edw5eaksgmrc9f2xwfrp` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499612, Requested 1034. Please try again in 1m51.6288s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Enriching chunks for Contextual Retrieval:   1%|          | 2/171 [00:02<03:10,  1.13s/it]

...

Error enriching chunk: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km59edw5eaksgmrc9f2xwfrp` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499539, Requested 1248. Please try again in 2m15.9936s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Enriching chunks for Contextual Retrieval:  70%|███████   | 120/171 [03:13<00:55,  1.09s/it]

Error enriching chunk: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01km59edw5eaksgmrc9f2xwfrp` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499532, Requested 1262. Please try again in 2m17.2032s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Enriching chunks for Contextual Retrieval:  70%|███████   | 120/171 [03:14<01:22,  1.62s/it]


KeyboardInterrupt: 

In [ ]:
VECTOR_DB_CONTEXTUAL = []

for chunk in tqdm(contextual_chunks, desc="Embedding contextual chunks"):
    emb = get_embedding(chunk)
    VECTOR_DB_CONTEXTUAL.append((chunk, emb))

print("Contextual Vector DB size:", len(VECTOR_DB_CONTEXTUAL))
print("Embedding dimension:", VECTOR_DB_CONTEXTUAL[0][1].shape)

Embedding contextual chunks: 100%|██████████| 171/171 [00:11<00:00, 14.98it/s]

Contextual Vector DB size: 171
Embedding dimension: (384,)


retrieval test for contextual DB

In [ ]:
test_question = "What is self-attention in a transformer?"
retrieved_contextual = retrieve(test_question, VECTOR_DB_CONTEXTUAL, top_k=3)

print("Question:", test_question)
print()

for i, (chunk, score) in enumerate(retrieved_contextual, 1):
    print(f"--- Contextual chunk {i} | score={score:.4f} ---")
    print(chunk[:1200])
    print()

Question: What is self-attention in a transformer?

--- Contextual chunk 1 | score=0.8002 ---
ding and all the outputs from all
the previous layers and blocks.
The core intuition of the transformer, and the component that distinguishes it
from the feedforward layers we saw in Chapter 6, is this multi-head attention layer,
also called a self-attention layer. Attention can be thought of as a way to build
contextual representations of a token’s meaning by attending to and integrating
information from surrounding tokens, helping the model learn how tokens relate to
each other over large span

--- Contextual chunk 2 | score=0.7966 ---
ther linear projection WO∈ RAdv×d to reshape it, resulting in the multi-head
attention vector ai with the correct output shape [1× d] at each input i.
8.2 Transformer Blocks
The self-attention calculation lies at the core of what’s called a transformer block,
which, in addition to the self-attention layer, includes three other kinds of layers: (1)
a feedforwar

answer function for contextual retrieval

In [ ]:
def answer_question_contextual(query, vector_db, top_k=3, model_name="llama-3.1-8b-instant"):
    retrieved = retrieve(query, vector_db, top_k=top_k)
    context = "\n\n".join([chunk for chunk, _ in retrieved])

    prompt = f"""
You are a helpful assistant answering questions strictly based on the provided context.

Context:
{context}

Question:
{query}

Instructions:
- Answer using only the provided context.
- If the answer is not found in the context, say: "The answer is not found in the provided context."
- Keep the answer concise, around 1-3 sentences.
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer = response.choices[0].message.content.strip()
    return answer, retrieved

test one contextual answer

In [ ]:
test_question = "What is self-attention in a transformer?"
test_answer_contextual, test_retrieved_contextual = answer_question_contextual(
    test_question, VECTOR_DB_CONTEXTUAL, top_k=3
)

print("Question:", test_question)
print("\nContextual Retrieval Answer:\n")
print(test_answer_contextual)

print("\nRetrieved contextual chunks:\n")
for i, (chunk, score) in enumerate(test_retrieved_contextual, 1):
    print(f"--- Chunk {i} | score={score:.4f} ---")
    print(chunk[:1200])
    print()

Question: What is self-attention in a transformer?

Contextual Retrieval Answer:

Self-attention in a transformer is a way to build contextual representations of a token's meaning by attending to and integrating information from surrounding tokens, helping the model learn how tokens relate to each other over large spans. It is a multi-head attention computation that takes an input vector and maps it to an output by adding in vectors from prior tokens, weighted by their relevance.

Retrieved contextual chunks:

--- Chunk 1 | score=0.8002 ---
ding and all the outputs from all
the previous layers and blocks.
The core intuition of the transformer, and the component that distinguishes it
from the feedforward layers we saw in Chapter 6, is this multi-head attention layer,
also called a self-attention layer. Attention can be thought of as a way to build
contextual representations of a token’s meaning by attending to and integrating
information from surrounding tokens, helping the model learn 

run contextual retrieval for all 20 questions

In [ ]:
for item in tqdm(qa_pairs, desc="Running Contextual Retrieval"):
    question = item["question"]

    try:
        answer, retrieved = answer_question_contextual(question, VECTOR_DB_CONTEXTUAL, top_k=3)
        item["contextual_retrieval_answer"] = answer
        item["contextual_retrieval_sources"] = [chunk for chunk, _ in retrieved]
    except Exception as e:
        item["contextual_retrieval_answer"] = f"ERROR: {str(e)}"
        item["contextual_retrieval_sources"] = []

    time.sleep(1)  # helps with rate limits

print("Finished generating Contextual Retrieval answers.")

Running Contextual Retrieval: 100%|██████████| 20/20 [02:12<00:00,  6.61s/it]

Finished generating Contextual Retrieval answers.


In [ ]:
with open("answer/qa_pairs_with_both_methods.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print("Saved results with Naive RAG and Contextual Retrieval.")

Saved results with Naive RAG and Contextual Retrieval.


quick comparison check

In [ ]:
print("nee")

nee


In [ ]:
for i in range(3):
    print(f"Q{i+1}: {qa_pairs[i]['question']}")
    print("Ground truth:", qa_pairs[i]["ground_truth_answer"])
    print("Naive RAG:", qa_pairs[i]["naive_rag_answer"])
    print("Contextual Retrieval:", qa_pairs[i]["contextual_retrieval_answer"])
    print("-" * 100)

Q1: What is the primary purpose of the self-attention mechanism in a transformer?
Ground truth: Self-attention allows a model to build contextual representations of a token by integrating information from other tokens in the sequence.
Naive RAG: The primary purpose of the self-attention mechanism in a transformer is to build contextual representations of a token's meaning by attending to and integrating information from surrounding tokens, helping the model learn how tokens relate to each other over large spans.
Contextual Retrieval: The primary purpose of the self-attention mechanism in a transformer is to build contextual representations of a token's meaning by attending to and integrating information from surrounding tokens, helping the model learn how tokens relate to each other over large spans.
----------------------------------------------------------------------------------------------------
Q2: In the attention mechanism, what roles do the query, key, and value vectors play?
G

In [ ]:
from rouge_score import rouge_scorer

with open("data/qa_pairs_with_both_methods.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

print("Loaded QA pairs:", len(qa_pairs))
print(qa_pairs[0].keys())

Loaded QA pairs: 20
dict_keys(['question', 'ground_truth_answer', 'naive_rag_answer', 'contextual_retrieval_answer', 'contextual_retrieval_sources'])


In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

In [ ]:
results = []

for item in qa_pairs:
    question = item["question"]
    ground_truth = item["ground_truth_answer"]
    naive_answer = item["naive_rag_answer"]
    contextual_answer = item["contextual_retrieval_answer"]

    naive_scores = scorer.score(ground_truth, naive_answer)
    contextual_scores = scorer.score(ground_truth, contextual_answer)

    results.append({
        "question": question,
        "ground_truth_answer": ground_truth,
        "naive_rag_answer": naive_answer,
        "contextual_retrieval_answer": contextual_answer,

        "naive_rouge1": naive_scores["rouge1"].fmeasure,
        "naive_rouge2": naive_scores["rouge2"].fmeasure,
        "naive_rougeL": naive_scores["rougeL"].fmeasure,

        "contextual_rouge1": contextual_scores["rouge1"].fmeasure,
        "contextual_rouge2": contextual_scores["rouge2"].fmeasure,
        "contextual_rougeL": contextual_scores["rougeL"].fmeasure,
    })

print("Computed ROUGE for all questions.")

Computed ROUGE for all questions.


In [ ]:
df_results = pd.DataFrame(results)
df_results.head()

,question,ground_truth_answer,naive_rag_answer,contextual_retrieval_answer,naive_rouge1,naive_rouge2,naive_rougeL,contextual_rouge1,contextual_rouge2,contextual_rougeL
0,What is the primary purpose of the self-attent...,Self-attention allows a model to build context...,The primary purpose of the self-attention mech...,The primary purpose of the self-attention mech...,0.593750,0.290323,0.500000,0.593750,0.290323,0.500000
1,"In the attention mechanism, what roles do the ...",The query represents the current token being c...,"The query, key, and value vectors play the fol...",ERROR: Error code: 429 - {'error': {'message':...,0.419753,0.151899,0.271605,0.108696,0.000000,0.065217
2,Why is a scaling factor used in the dot produc...,The dot product is scaled by the square root o...,Exponentiating large values can lead to numeri...,ERROR: Error code: 429 - {'error': {'message':...,0.493151,0.366197,0.328767,0.023256,0.000000,0.023256
3,Why do transformers use multi-head attention i...,Multi-head attention allows the model to atten...,Transformers use multi-head attention instead ...,ERROR: Error code: 429 - {'error': {'message':...,0.472222,0.142857,0.250000,0.120482,0.000000,0.072289
4,What components are included in a standard tra...,A transformer block includes a multi-head self...,A standard transformer block consists of a res...,ERROR: Error code: 429 - {'error': {'message':...,0.447761,0.215385,0.388060,0.000000,0.000000,0.000000


In [ ]:
df_results = pd.DataFrame(results)
df_results.head()

,question,ground_truth_answer,naive_rag_answer,contextual_retrieval_answer,naive_rouge1,naive_rouge2,naive_rougeL,contextual_rouge1,contextual_rouge2,contextual_rougeL
0,What is the primary purpose of the self-attent...,Self-attention allows a model to build context...,The primary purpose of the self-attention mech...,The primary purpose of the self-attention mech...,0.593750,0.290323,0.500000,0.593750,0.290323,0.500000
1,"In the attention mechanism, what roles do the ...",The query represents the current token being c...,"The query, key, and value vectors play the fol...",ERROR: Error code: 429 - {'error': {'message':...,0.419753,0.151899,0.271605,0.108696,0.000000,0.065217
2,Why is a scaling factor used in the dot produc...,The dot product is scaled by the square root o...,Exponentiating large values can lead to numeri...,ERROR: Error code: 429 - {'error': {'message':...,0.493151,0.366197,0.328767,0.023256,0.000000,0.023256
3,Why do transformers use multi-head attention i...,Multi-head attention allows the model to atten...,Transformers use multi-head attention instead ...,ERROR: Error code: 429 - {'error': {'message':...,0.472222,0.142857,0.250000,0.120482,0.000000,0.072289
4,What components are included in a standard tra...,A transformer block includes a multi-head self...,A standard transformer block consists of a res...,ERROR: Error code: 429 - {'error': {'message':...,0.447761,0.215385,0.388060,0.000000,0.000000,0.000000


In [ ]:
summary = pd.DataFrame([
    {
        "Method": "Naive RAG",
        "ROUGE-1": df_results["naive_rouge1"].mean(),
        "ROUGE-2": df_results["naive_rouge2"].mean(),
        "ROUGE-L": df_results["naive_rougeL"].mean(),
    },
    {
        "Method": "Contextual Retrieval",
        "ROUGE-1": df_results["contextual_rouge1"].mean(),
        "ROUGE-2": df_results["contextual_rouge2"].mean(),
        "ROUGE-L": df_results["contextual_rougeL"].mean(),
    }
])

summary

,Method,ROUGE-1,ROUGE-2,ROUGE-L
0,Naive RAG,0.394312,0.141468,0.288752
1,Contextual Retrieval,0.110405,0.020516,0.085793


In [ ]:
summary_rounded = summary.copy()
summary_rounded["ROUGE-1"] = summary_rounded["ROUGE-1"].round(4)
summary_rounded["ROUGE-2"] = summary_rounded["ROUGE-2"].round(4)
summary_rounded["ROUGE-L"] = summary_rounded["ROUGE-L"].round(4)

summary_rounded

,Method,ROUGE-1,ROUGE-2,ROUGE-L
0,Naive RAG,0.3943,0.1415,0.2888
1,Contextual Retrieval,0.1104,0.0205,0.0858


In [ ]:
df_results.to_csv("answer/rouge_detailed_results.csv", index=False)
summary_rounded.to_csv("answer/rouge_summary.csv", index=False)

print("Saved ROUGE detailed results and summary.")

Saved ROUGE detailed results and summary.


Sav Broken API answer

In [ ]:
failed_rows = []

for i, item in enumerate(qa_pairs):
    ans = item.get("contextual_retrieval_answer", "")
    
    if "ERROR" in ans or ans.strip() == "":
        failed_rows.append(i)

print("Failed rows:", len(failed_rows))
print(failed_rows[:10])

In [59]:
print("hello")

hello
